In [1]:
import datasets

data = datasets.load_dataset("dair-ai/emotion", "split")

/Users/okorovii/Letyshops/entity-recognition/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from collections import defaultdict

c = defaultdict(int)
id_to_label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

for i in data['train']:
    c[id_to_label[i['label']]] += 1

for i in data['test']:
    c[id_to_label[i['label']]] += 1

c

defaultdict(int,
            {'sadness': 5247,
             'anger': 2434,
             'love': 1463,
             'surprise': 638,
             'fear': 2161,
             'joy': 6057})

In [12]:
c['neutral'] = 3000 + 1103
c['sadness'] = 3144
c['joy'] = 3057
c['surprise'] = 1638
c

defaultdict(int,
            {'sadness': 3144,
             'anger': 2434,
             'love': 1463,
             'surprise': 1638,
             'fear': 2161,
             'joy': 3057,
             'neutral': 4103})

In [13]:
sum(c.values())

18000

In [3]:
# dump text data into txt file

def dump_text_data(data, filename):
    with open(filename, "w") as f:
        for item in data:
            f.write(item['text'] + "\n")

dump_text_data(data['train'], 'data/emotion_train.txt')
dump_text_data(data['test'], 'data/emotion_test.txt')
dump_text_data(data['validation'], 'data/emotion_validation.txt')


In [4]:
from mlx_lm import load, generate

model, tokenizer = load("lang-uk/dragoman-4bit")
response = generate(model, tokenizer, prompt="[INST] who holds this neighborhood? [/INST]", verbose=True)
print(response)

Fetching 7 files: 100%|██████████| 7/7 [01:49<00:00, 15.67s/it]


хто тримає цей район?
Prompt: 13 tokens, 21.334 tokens-per-sec
Generation: 10 tokens, 28.663 tokens-per-sec
Peak memory: 4.328 GB
хто тримає цей район?


In [13]:
import tqdm
data_val_ukr = []
for item in tqdm.tqdm(data['validation']):
    translated = generate(model, tokenizer, prompt=f"[INST] {item['text']} [/INST]", verbose=False)
    data_val_ukr.append({"text": translated, "label": item['label']})



100%|██████████| 2000/2000 [56:45<00:00,  1.70s/it] 


In [6]:
import string
import json

def convert_json_to_fasttext(json_path, output_path):
    """
    Convert emotion JSON data to FastText format.
    Cleans text by removing punctuation.
    Each line format: __label__X text
    """
    with open(json_path, 'r') as f:
        data = json.load(f)

    with open(output_path, 'w') as f:
        for item in data:
            # Remove punctuation and extra whitespace
            cleaned_text = item['text'].translate(str.maketrans('', '', string.punctuation)).strip().lower()
            # FastText format: __label__X text
            f.write(f"__label__{item['label']} {cleaned_text}\n")



In [7]:
def create_text_corpus(*json_paths, output_path):
    """
    Create a text corpus from multiple JSON files.
    Cleans text by removing punctuation.
    """
    with open(output_path, 'w') as f:
        for json_path in json_paths:
            with open(json_path, 'r') as f_json:
                data = json.load(f_json)
                for item in data:
                    # Remove punctuation and extra whitespace
                    cleaned_text = item['text'].translate(str.maketrans('', '', string.punctuation)).strip().lower()
                    f.write(f"{cleaned_text}\n")


create_text_corpus('data/emotion/emotion_train_ukr.json', 'data/emotion/emotion_test_ukr.json', output_path='data/emotion/emotion_corpus_ukr.txt')

In [8]:
# Usage example
convert_json_to_fasttext('data/emotion/emotion_train_ukr.json', 'data/emotion/emotion_train_ukr.txt')
convert_json_to_fasttext('data/emotion/emotion_test_ukr.json', 'data/emotion/emotion_test_ukr.txt')

In [1]:
# unsuperwised learning
import fasttext

model = fasttext.train_unsupervised(input="data/emotion/emotion_corpus_ukr.txt", model="skipgram", epoch=100, lr=0.1, verbose=2, minCount=1, bucket=200000, dim=100, minn=3, maxn=5)

with open('data/emotion/ukrtext.vec', 'w') as f:
    f.write(f"{len(model.get_words())} {model.get_dimension()}\n")
    for word in model.get_words():
        f.write(f"{word} {' '.join(map(str, model.get_word_vector(word)))}\n")

Read 0M words
Number of words:  29408
Number of labels: 0
Progress: 100.0% words/sec/thread:  190768 lr:  0.000000 avg.loss:  1.278143 ETA:   0h 0m 0s


In [2]:
import fasttext

# Train the model
model = fasttext.train_supervised(input="data/emotion/emotion_train_ukr.txt", epoch=50, lr=0.1, verbose=2, minCount=1, bucket=200000, dim=100, minn=3, maxn=5,  pretrainedVectors="data/emotion/ukrtext.vec")

Read 0M words
Number of words:  27505
Number of labels: 6
Progress: 100.0% words/sec/thread:  764390 lr:  0.000000 avg.loss:  0.306409 ETA:   0h 0m 0s


In [2]:
# sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5).
id_to_label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

In [3]:
def print_results(model, input_path, k):
    num_records, precision_at_k, recall_at_k = model.test(input_path, k)
    f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)

    print("records\t{}".format(num_records))
    print("Precision@{}\t{:.3f}".format(k, precision_at_k))
    print("Recall@{}\t{:.3f}".format(k, recall_at_k))
    print("F1@{}\t{:.3f}".format(k, f1_at_k))
    print()

print_results(model, 'data/emotion/emotion_test_ukr.txt', k=1)

records	2000
Precision@1	0.735
Recall@1	0.735
F1@1	0.735



In [26]:
%%timeit
model.predict("Я дуже радий, що ти прийшов на мій день народження")

AttributeError: 'XLMRobertaForSequenceClassification' object has no attribute 'predict'

In [5]:
def calculate_accuracy(model, test_path, id_to_label):
    """
    Calculate accuracy for FastText model predictions.
    Returns overall accuracy and per-class accuracies.
    """
    correct = 0
    total = 0
    class_correct = {i: 0 for i in id_to_label}
    class_total = {i: 0 for i in id_to_label}

    with open(test_path, 'r') as f:
        for line in f:
            # Split line into label and text
            parts = line.strip().split(' ', 1)
            if len(parts) != 2:
                continue

            true_label = int(parts[0].replace('__label__', ''))
            text = parts[1]

            # Get prediction
            pred = model.predict(text)[0][0]
            pred_label = int(pred.replace('__label__', ''))

            # Update counters
            total += 1
            class_total[true_label] += 1
            if pred_label == true_label:
                correct += 1
                class_correct[true_label] += 1

    # Calculate accuracies
    overall_accuracy = correct / total
    class_accuracies = {id_to_label[i]: class_correct[i]/class_total[i]
                       for i in id_to_label if class_total[i] > 0}

    # Print results
    print(f"Overall Accuracy: {overall_accuracy:.3f}")
    print("\nPer-class Accuracies:")
    for emotion, acc in class_accuracies.items():
        print(f"{emotion}: {acc:.3f}")

    return overall_accuracy, class_accuracies

# Usage with your existing model
calculate_accuracy(model, 'data/emotion/emotion_test_ukr.txt', id_to_label)

Overall Accuracy: 0.735

Per-class Accuracies:
sadness: 0.804
joy: 0.837
love: 0.534
anger: 0.636
fear: 0.599
surprise: 0.506


(0.7345,
 {'sadness': 0.8036363636363636,
  'joy': 0.8366477272727273,
  'love': 0.5337078651685393,
  'anger': 0.6363636363636364,
  'fear': 0.5990566037735849,
  'surprise': 0.5061728395061729})

In [12]:
def count_fasttext_parameters(model):
    """
    Calculate the number of parameters in FastText model.
    """
    # Get model dimensions
    vocab_size = len(model.get_words())
    embedding_dim = model.get_dimension()
    output_dim = len(model.get_labels())

    # Calculate parameters
    embedding_params = vocab_size * embedding_dim
    output_layer_params = embedding_dim * output_dim
    total_params = embedding_params + output_layer_params

    # Print results
    print(f"Vocabulary size: {vocab_size:,}")
    print(f"Embedding dimension: {embedding_dim}")
    print(f"Number of classes: {output_dim}")
    print(f"\nParameters:")
    print(f"Embedding layer: {embedding_params:,}")
    print(f"Output layer: {output_layer_params:,}")
    print(f"Total parameters: {total_params:,}")

    return total_params

# Usage
total_params = count_fasttext_parameters(model)

Vocabulary size: 29,408
Embedding dimension: 100
Number of classes: 6

Parameters:
Embedding layer: 2,940,800
Output layer: 600
Total parameters: 2,941,400


Device set to use cpu


[[{'label': 'JOY', 'score': 0.9934096932411194},
  {'label': 'LOVE', 'score': 0.005158454645425081},
  {'label': 'ANGER', 'score': 0.0006234814063645899},
  {'label': 'SURPRISE', 'score': 0.0004272169608157128},
  {'label': 'SADNESS', 'score': 0.00021990039385855198},
  {'label': 'FEAR', 'score': 0.00016116451297421008}]]

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/Users/okorovii/Letyshops/entity-recognition/venv/lib/python3.12/site-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [5]:
model("hello my dear friend")

[[{'label': 'NEGATIVE', 'score': 0.0006886933697387576},
  {'label': 'POSITIVE', 'score': 0.9993113279342651}]]